# DeepFish marine detector — YOLOv8n (Kaggle T4)

Replaces the freshwater-trained detector with one trained on **DeepFish** (YOLO-Fish
benchmark, single-class "fish"), and proves the domain-gap fix with a **before/after** on the
held-out benchmark TEST split. Runs on Kaggle T4 — not locally.

- Data: DeepFish YOLO export (public GDrive) using the **benchmark-official split** (no re-split).
- Backgrounds: NOAA "Labeled Fishes in the Wild" negatives as hard-negatives in TRAIN only (~<=10%).
- Label format was verified locally (no conversion): `outputs/label_check.png`.

In [ ]:
# B1 — setup. CRITICAL: do NOT let pip replace Kaggle's GPU-matched torch/torchvision.
# A naive `pip install ultralytics` pulls a torch wheel whose CUDA arch may not match the
# assigned GPU -> "CUDA error: no kernel image is available for execution on the device".
# Pin the pre-installed torch/torchvision in a constraints file so the resolver can't swap them.
import glob
import os
import random
import shutil
import subprocess
import sys
from pathlib import Path

import torch
import torchvision
import yaml

Path("constraints.txt").write_text(
    f"torch=={torch.__version__}\ntorchvision=={torchvision.__version__}\n"
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "-c", "constraints.txt", "ultralytics", "albumentations", "gdown"],
    check=True,
)

# GPU diagnostic — confirm torch STILL matches the assigned GPU after the install.
print("torch              :", torch.__version__)
print("torch.version.cuda :", torch.version.cuda)
print("cuda available     :", torch.cuda.is_available())
print("device name        :", torch.cuda.get_device_name(0))
print("device capability  :", torch.cuda.get_device_capability(0))
print("arch list          :", torch.cuda.get_arch_list())

from ultralytics import YOLO

SEED = 42
random.seed(SEED)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("device", DEVICE)

WORK = Path("/kaggle/working")
RAW = WORK / "deepfish_raw"
DATASET = WORK / "deepfish"          # assembled YOLO dataset root
RAW.mkdir(parents=True, exist_ok=True)
for sub in ["images/train", "images/val", "images/test",
            "labels/train", "labels/val", "labels/test"]:
    (DATASET / sub).mkdir(parents=True, exist_ok=True)

In [ ]:
# A1 — fetch the annotated DeepFish export (YOLO-Fish benchmark, public GDrive ~1.1 GB).
# This single archive holds EVERY labeled clip (with its own train/ + valid/ subfolders) plus
# Deepfish/Nagative_samples. We re-split it CLIP-DISJOINT below, so the separate official test
# zip is NOT downloaded: its frame-level split shares clips with train -> temporal leak.
import gdown

FULL_ID = "10Pr4lLeSGTfkjA40ReGSC8H3a9onfMZ0"   # annotated DeepFish (all images + YOLO .txt labels)
full_zip = RAW / "deepfish_full.zip"
if not full_zip.exists():
    gdown.download(f"https://drive.google.com/uc?id={FULL_ID}", str(full_zip), quiet=False)
    # If gdown hits a Google quota error, mirror the zip to a Kaggle Dataset, attach it, and
    # point full_zip at the /kaggle/input/... path instead.

(RAW / "full").mkdir(exist_ok=True)
shutil.unpack_archive(str(full_zip), str(RAW / "full"))
print("unpacked annotated DeepFish export ->", RAW / "full")

In [ ]:
# A2 — assemble a CLIP-DISJOINT YOLO dataset from the annotated export.
# Every box-labeled frame is named "<clip>_f######" and belongs to one of ~46 clips. The
# dataset's OWN split is frame-level WITHIN those clips (train/ + valid/ share the same clips),
# so neighbouring near-identical frames straddle the split -> leak. We therefore IGNORE that
# split and partition WHOLE clips into train/val/test. Only box-labeled frames define the
# splits (guarantees boxes>0); NOAA backgrounds are added to TRAIN in the next cell. Class: fish.
import re
from collections import defaultdict

def img_index(root):
    """stem -> image path (recursive)."""
    return {Path(p).stem: p for p in glob.glob(str(Path(root) / "**" / "*.jpg"), recursive=True)}

def lbl_index(root):
    """stem -> label .txt path (recursive)."""
    return {Path(p).stem: p for p in glob.glob(str(Path(root) / "**" / "*.txt"), recursive=True)}

def clip_of(stem):
    """Clip/habitat id = filename with the trailing frame index '_f######' stripped."""
    return re.sub(r"_f\d+$", "", stem)

full_imgs = img_index(RAW / "full")
full_lbls = lbl_index(RAW / "full")

def n_boxes(stem):
    p = full_lbls.get(stem)
    if not p or not os.path.exists(p):
        return 0
    return sum(1 for ln in open(p).read().splitlines() if ln.strip())

# SOURCE DIAG — proves the label lookup resolves and where the boxes are.
full_exts = sorted({Path(p).suffix.lower()
                    for p in glob.glob(str(RAW / "full" / "**" / "*.*"), recursive=True)})
box_stems = [s for s in full_imgs if n_boxes(s) > 0]
box_clips = {clip_of(s) for s in box_stems}
print("=== SOURCE ANNOTATION DIAG ===")
print(f"full export: images={len(full_imgs)} txt={len(full_lbls)} exts={full_exts}")
print(f"  img∩lbl stems = {len(set(full_imgs) & set(full_lbls))} / {len(full_imgs)}")
print(f"  box-labeled frames = {len(box_stems)} across {len(box_clips)} clips")

# Partition the labeled clips by WHOLE clip (deterministic).
by_clip = defaultdict(list)
for s in box_stems:
    by_clip[clip_of(s)].append(s)
clips = sorted(by_clip)
random.Random(SEED).shuffle(clips)
n = len(clips)
n_test = max(1, round(0.20 * n))      # ~20% of CLIPS held out for TEST
n_val = max(1, round(0.12 * n))       # ~12% of CLIPS for VAL
test_clips = set(clips[:n_test])
val_clips = set(clips[n_test:n_test + n_val])
tr_clips = set(clips[n_test + n_val:])
split_of = {c: "train" for c in tr_clips}
split_of.update({c: "val" for c in val_clips})
split_of.update({c: "test" for c in test_clips})

def place(stem, split):
    """Copy frame to images/<split>/ AND its label to labels/<split>/ with the SAME stem."""
    shutil.copy(full_imgs[stem], DATASET / "images" / split / f"{stem}.jpg")
    src_txt = full_lbls.get(stem)
    dst_txt = DATASET / "labels" / split / f"{stem}.txt"
    if src_txt and os.path.exists(src_txt):
        shutil.copy(src_txt, dst_txt)
    else:
        dst_txt.write_text("")

for s in box_stems:
    place(s, split_of[clip_of(s)])

# HARD GATE 1: clip-disjoint.
assert tr_clips.isdisjoint(val_clips), f"LEAK train∩val {tr_clips & val_clips}"
assert tr_clips.isdisjoint(test_clips), f"LEAK train∩test {tr_clips & test_clips}"
assert val_clips.isdisjoint(test_clips), f"LEAK val∩test {val_clips & test_clips}"

# HARD GATE 2: image<->label pairing AND non-empty labels.
def verify(split):
    ish = {Path(p).stem for p in glob.glob(str(DATASET / "images" / split / "*.jpg"))}
    lps = glob.glob(str(DATASET / "labels" / split / "*.txt"))
    lsh = {Path(p).stem for p in lps}
    boxes = sum(len([ln for ln in open(p).read().splitlines() if ln.strip()]) for p in lps)
    return ish, lsh, boxes

print("split = CLIP-DISJOINT (whole clips partitioned; official frame-level split NOT used)")
print(f"layout: {DATASET}/images/<split>/<stem>.jpg  <->  {DATASET}/labels/<split>/<stem>.txt")
for sp in ["train", "val", "test"]:
    ish, lsh, b = verify(sp)
    miss, orph = ish - lsh, lsh - ish
    c = len({clip_of(s) for s in ish})
    print(f"{sp:5s}: images={len(ish):5d} labels={len(lsh):5d} "
          f"missing={len(miss)} orphan={len(orph)} boxes={b:6d} clips={c}")
    assert len(ish) == len(lsh), f"{sp}: image/label count mismatch {len(ish)} vs {len(lsh)}"
    assert not miss and not orph, f"{sp}: pairing broken miss={len(miss)} orph={len(orph)}"
    assert b > 0, f"{sp}: 0 box-lines across {len(lsh)} labels — labels empty"
print(f"clips: train={len(tr_clips)} val={len(val_clips)} test={len(test_clips)}")

In [ ]:
# A2 — NOAA "Labeled Fishes in the Wild" negatives as hard-negative backgrounds (TRAIN only).
# Cap at <=10% of total images; report the actual background fraction.
NOAA_URL = ("https://storage.googleapis.com/nmfs_odp_swfsc/"
            "Fisheries%20Resources%20Division/Labeled_Fishes_In_The_Wild.zip")
noaa_zip = RAW / "noaa.zip"
if not noaa_zip.exists():
    subprocess.run(["wget", "-q", "-O", str(noaa_zip), NOAA_URL], check=True)
(RAW / "noaa").mkdir(exist_ok=True)
shutil.unpack_archive(str(noaa_zip), str(RAW / "noaa"))

# Locate the negative (no-fish) seabed images inside the archive.
all_imgs = [p for p in glob.glob(str(RAW / "noaa" / "**" / "*.*"), recursive=True)
            if p.lower().endswith((".jpg", ".jpeg", ".png"))]
neg = [p for p in all_imgs if "negativ" in p.lower()]
if not neg:
    print("WARNING: no path matched 'negativ'. Archive image dirs:")
    dirs = sorted({str(Path(p).parent) for p in all_imgs})
    for d in dirs[:20]:
        print("  ", d)
print(f"NOAA negative images found: {len(neg)}")

n_tr = len(glob.glob(str(DATASET / "images" / "train" / "*.jpg")))
n_va = len(glob.glob(str(DATASET / "images" / "val" / "*.jpg")))
n_te = len(glob.glob(str(DATASET / "images" / "test" / "*.jpg")))
total_fish = n_tr + n_va + n_te
cap = int(0.10 * total_fish)
use = neg[: max(0, min(len(neg), cap))]
for i, p in enumerate(use):
    name = f"noaa_neg_{i:04d}.jpg"
    shutil.copy(p, DATASET / "images" / "train" / name)
    (DATASET / "labels" / "train" / f"noaa_neg_{i:04d}.txt").write_text("")  # empty = background

total_all = total_fish + len(use)
frac = 100 * len(use) / max(1, total_all)
print(f"added {len(use)} NOAA backgrounds to TRAIN (cap {cap} = 10% of {total_fish} fish images)")
print(f"background fraction: {len(use)}/{total_all} = {frac:.1f}% of all images")

In [ ]:
# A3 — data.yaml (single class). Benchmark TEST stays held out for the final number.
data_yaml = {
    "path": str(DATASET),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 1,
    "names": ["fish"],
}
yaml_path = WORK / "deepfish.yaml"
yaml_path.write_text(yaml.safe_dump(data_yaml, sort_keys=False))
print(yaml_path.read_text())

In [ ]:
# B2 — BEFORE: current freshwater detector on the held-out DeepFish TEST set.
# yolo_fish.pt is attached via a private Kaggle dataset; find it wherever it got mounted
# under /kaggle/input (folder name varies if the dataset is renamed/forked).
_fw = sorted(glob.glob("/kaggle/input/**/yolo_fish.pt", recursive=True))
assert _fw, (
    "freshwater weights not found: no yolo_fish.pt under /kaggle/input — "
    "attach the freshwater-fish-detector dataset"
)
FRESHWATER_PT = _fw[0]
print("freshwater weights:", FRESHWATER_PT)

before = YOLO(FRESHWATER_PT).val(
    data=str(yaml_path), split="test", imgsz=640, device=DEVICE,
    project=str(WORK / "runs"), name="before_freshwater", verbose=False,
)
before_map50, before_map = before.box.map50, before.box.map
print(f"BEFORE (freshwater) on DeepFish TEST: mAP50={before_map50:.4f}  mAP50-95={before_map:.4f}")
print("Expect this to be poor — that poorness IS the domain-gap evidence.")

In [ ]:
# B3 — TRAIN from COCO-pretrained yolov8n (NOT the freshwater checkpoint -> no freshwater bias).
model = YOLO("yolov8n.pt")
train_res = model.train(
    data=str(yaml_path),
    imgsz=640,
    epochs=100,
    patience=20,
    batch=-1,        # auto-batch for the T4
    amp=True,
    seed=SEED,
    device=DEVICE,
    cache=False,     # do NOT cache to RAM — multi-GB image set OOMs Kaggle's ~13 GB
    hsv_s=0.9,       # raise saturation/value jitter to fight the blue/green water cast
    hsv_v=0.6,
    project=str(WORK / "runs"),
    name="deepfish_yolov8n",
)
best_pt = Path(train_res.save_dir) / "weights" / "best.pt"
print("best.pt:", best_pt)

In [ ]:
# B4 — AFTER: eval new best.pt on carved val and held-out benchmark TEST; NOAA background FP count.
new = YOLO(str(best_pt))
val_m = new.val(data=str(yaml_path), split="val", imgsz=640, device=DEVICE,
                project=str(WORK / "runs"), name="after_val", verbose=False)
test_m = new.val(data=str(yaml_path), split="test", imgsz=640, device=DEVICE,
                 project=str(WORK / "runs"), name="after_test", verbose=False)

def row(m):
    return m.box.map50, m.box.map, m.box.mp, m.box.mr

v = row(val_m)
t = row(test_m)
print(f"AFTER  val : mAP50={v[0]:.4f} mAP50-95={v[1]:.4f} P={v[2]:.4f} R={v[3]:.4f}")
print(f"AFTER  TEST: mAP50={t[0]:.4f} mAP50-95={t[1]:.4f} P={t[2]:.4f} R={t[3]:.4f}")

# Background false positives: predict on the NOAA negatives (should be ~0 — what hard-negatives buy).
bg_imgs = glob.glob(str(DATASET / "images" / "train" / "noaa_neg_*.jpg"))
fp = 0
if bg_imgs:
    for r in new.predict(bg_imgs, imgsz=640, device=DEVICE, conf=0.25, verbose=False):
        fp += len(r.boxes)
    print(f"NOAA background frames: {len(bg_imgs)} | total false-positive boxes: {fp}")
else:
    print("No NOAA background frames present to test.")

In [ ]:
# B5 — RESULTS. Same CLIP-DISJOINT held-out TEST (whole clips unseen in train), before vs after.
print("========== BEFORE / AFTER (clip-disjoint held-out TEST) ==========")
print(f'{"detector":26s} {"mAP50":>8s} {"mAP50-95":>9s}')
print(f'{"freshwater (before)":26s} {before_map50:8.4f} {before_map:9.4f}')
print(f'{"DeepFish yolov8n (after)":26s} {t[0]:8.4f} {t[1]:9.4f}')
print(f"delta mAP50: {t[0] - before_map50:+.4f}")
print()
print("NOTE on comparability:")
print("  This TEST is a CLIP-DISJOINT re-split of DeepFish, NOT the official YOLO-Fish")
print("  frame-level split (which leaks near-identical neighbouring frames across train/test).")
print("  So this mAP is an honest no-leak generalization estimate and is deliberately NOT")
print("  comparable to the published DeepFish AP (~0.76, measured on the leaky split).")
print("  The before/after delta is valid: both detectors are scored on the same held-out TEST.")

In [ ]:
# B6 — export new detector + keep run artifacts.
OUT = WORK / "export"
OUT.mkdir(exist_ok=True)
shutil.copy(best_pt, OUT / "yolo_fish.pt")
print("best.pt copied to", OUT / "yolo_fish.pt")
print("run artifacts (PR_curve.png, confusion_matrix.png, results.png) in:", best_pt.parent.parent)

## Download
1. Output tab → `/kaggle/working/export/yolo_fish.pt` → download → place locally at `weights/yolo_fish.pt`.
2. Also grab `runs/deepfish_yolov8n/{PR_curve.png, confusion_matrix.png, results.png}` for the writeup.
3. Local real-time picks it up automatically (`realtime.py --detector` default = `weights/yolo_fish.pt`).